# EfficientNet-B0)) validation 학습 opencv

In [15]:
import torch

# 파이썬 pickle 대신 PyTorch의 torch.load를 사용합니다.
# 파일명이 'optimized_bee_model_backup.pth'이므로 경로를 올바르게 맞춰줍니다.
model_data = torch.load('optimized_bee_model.pth', map_location='cpu')

print(model_data.keys())  # 내부 레이어 이름 확인

odict_keys(['conv_stem.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'blocks.0.0.conv_dw.weight', 'blocks.0.0.bn1.weight', 'blocks.0.0.bn1.bias', 'blocks.0.0.bn1.running_mean', 'blocks.0.0.bn1.running_var', 'blocks.0.0.bn1.num_batches_tracked', 'blocks.0.0.se.conv_reduce.weight', 'blocks.0.0.se.conv_reduce.bias', 'blocks.0.0.se.conv_expand.weight', 'blocks.0.0.se.conv_expand.bias', 'blocks.0.0.conv_pw.weight', 'blocks.0.0.bn2.weight', 'blocks.0.0.bn2.bias', 'blocks.0.0.bn2.running_mean', 'blocks.0.0.bn2.running_var', 'blocks.0.0.bn2.num_batches_tracked', 'blocks.1.0.conv_pw.weight', 'blocks.1.0.bn1.weight', 'blocks.1.0.bn1.bias', 'blocks.1.0.bn1.running_mean', 'blocks.1.0.bn1.running_var', 'blocks.1.0.bn1.num_batches_tracked', 'blocks.1.0.conv_dw.weight', 'blocks.1.0.bn2.weight', 'blocks.1.0.bn2.bias', 'blocks.1.0.bn2.running_mean', 'blocks.1.0.bn2.running_var', 'blocks.1.0.bn2.num_batches_tracked', 'blocks.1.0.se.conv_reduce.weigh

In [17]:
import torch
import timm

# 1. 모델 아키텍처 정의
# 저장된 파일의 구조와 정확히 일치하도록 num_classes를 설정해야 합니다.
# (파일이 4개 클래스용이므로 4로 지정)
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=4)

# 2. 가중치 파일 경로 설정
# 사용하고 싶은 파일명을 입력하세요 ('optimized_bee_model.pth' 또는 '..._backup.pth')
model_path = 'optimized_bee_model.pth' 

# 3. 가중치 로드
try:
    # map_location='cpu'를 사용하여 CPU 환경에서도 안전하게 불러옵니다.
    model_data = torch.load(model_path, map_location='cpu')
    
    # 모델에 가중치 입히기
    model.load_state_dict(model_data)
    print(f"성공: '{model_path}'의 가중치가 모델에 정상적으로 로드되었습니다.")
    
except Exception as e:
    print(f"오류 발생: 가중치 로드 실패 - {e}")

# 4. 모델 모드 설정
# 추론(테스트)을 위해 모델을 eval 모드로 전환합니다.
model.eval()

# 5. 확인 출력 (선택 사항)
# 모델이 정상적으로 로드되었는지 최종 확인
print("모델이 추론 준비 완료 상태입니다.")

# 이후 이 model 객체를 사용하여 실제 이미지를 예측(predict)할 수 있습니다.

성공: 'optimized_bee_model.pth'의 가중치가 모델에 정상적으로 로드되었습니다.
모델이 추론 준비 완료 상태입니다.


In [18]:
from PIL import Image
from timm.data import create_transform
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD

# 전처리 함수 정의 (이미지 크기 조절 및 정규화)
transform = create_transform(
    input_size=224, # efficientnet_b0의 기본 입력 사이즈
    mean=IMAGENET_DEFAULT_MEAN,
    std=IMAGENET_DEFAULT_STD
)

In [19]:
def predict_image(image_path, model):
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0) # 배치 차원 추가 (1, 3, 224, 224)
    
    with torch.no_grad(): # 역전파 비활성화 (메모리 절약)
        output = model(img_tensor)
        prediction = torch.argmax(output, dim=1)
        
    return prediction.item()

# 사용 예시
image_path = 'test.jpg' # 실제 이미지 파일 경로
result = predict_image(image_path, model)
print(f"예측된 클래스 인덱스: {result}")

예측된 클래스 인덱스: 2


In [20]:
# 테스트할 이미지 파일 경로들을 리스트로 만듭니다.
# 예: 폴더별로 한 장씩 골라보세요.
test_images = ['test.jpg', 'test1.jpg', 'test2.jpg','test3.jpg']

for img_path in test_images:
    idx = predict_image(img_path, model)
    print(f"이미지 {img_path} -> 예측된 인덱스: {idx}")

이미지 test.jpg -> 예측된 인덱스: 2
이미지 test1.jpg -> 예측된 인덱스: 3
이미지 test2.jpg -> 예측된 인덱스: 1
이미지 test3.jpg -> 예측된 인덱스: 0


# opencv

# -- 응애 분류

In [21]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import ImageFont, ImageDraw, Image

# 1. 이미지 로드 (이 셀 내에서 한꺼번에 처리)
image_path = 'test.jpg' 
img_cv = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
img_pil = Image.fromarray(img_rgb)

# 2. 전처리 (이전 셀에서 정의한 transform 사용)
img_tensor = transform(img_pil).unsqueeze(0)

# 3. 모델 추론
model.eval()
with torch.no_grad():
    output = model(img_tensor)
    probabilities = F.softmax(output[0], dim=0)
    prediction = torch.argmax(output, dim=1).item()
    confidence = probabilities[prediction].item() * 100

# --- 추가된 출력 부분 ---
labels = ["알", "백묵병", "응애", "정상"]
print("-" * 30)
print(f"분석 결과: {labels[prediction]}")
print(f"확신도(Confidence): {confidence:.2f}%")
print("-" * 30)
# ----------------------

# 4. 한글 및 퍼센트 출력 준비 (이미지용)
fontpath = "malgun.ttf" 
font = ImageFont.truetype(fontpath, 40)
draw = ImageDraw.Draw(img_pil)
text = f"{labels[prediction]} ({confidence:.1f}%)"
draw.text((50, 50), text, font=font, fill=(0, 255, 0))

# 5. OpenCV로 결과 출력
img_result = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
cv2.imshow('Bee Detection', img_result)
cv2.waitKey(0)
cv2.destroyAllWindows()

------------------------------
분석 결과: 응애
확신도(Confidence): 100.00%
------------------------------


In [22]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import ImageFont, ImageDraw, Image

# 1. 이미지 로드 (이 셀 내에서 한꺼번에 처리)
image_path = 'test4.jpg' 
img_cv = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
img_pil = Image.fromarray(img_rgb)

# 2. 전처리 (이전 셀에서 정의한 transform 사용)
img_tensor = transform(img_pil).unsqueeze(0)

# 3. 모델 추론
model.eval()
with torch.no_grad():
    output = model(img_tensor)
    probabilities = F.softmax(output[0], dim=0)
    prediction = torch.argmax(output, dim=1).item()
    confidence = probabilities[prediction].item() * 100

# --- 추가된 출력 부분 ---
labels = ["알", "백묵병", "응애", "정상"]
print("-" * 30)
print(f"분석 결과: {labels[prediction]}")
print(f"확신도(Confidence): {confidence:.2f}%")
print("-" * 30)
# ----------------------

# 4. 한글 및 퍼센트 출력 준비 (이미지용)
fontpath = "malgun.ttf" 
font = ImageFont.truetype(fontpath, 40)
draw = ImageDraw.Draw(img_pil)
text = f"{labels[prediction]} ({confidence:.1f}%)"
draw.text((50, 50), text, font=font, fill=(0, 255, 0))

# 5. OpenCV로 결과 출력
img_result = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
cv2.imshow('Bee Detection', img_result)
cv2.waitKey(0)
cv2.destroyAllWindows()

------------------------------
분석 결과: 정상
확신도(Confidence): 99.63%
------------------------------


In [8]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import ImageFont, ImageDraw, Image

# 1. 이미지 로드 (이 셀 내에서 한꺼번에 처리)
image_path = 'test3.jpg' 
img_cv = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
img_pil = Image.fromarray(img_rgb)

# 2. 전처리 (이전 셀에서 정의한 transform 사용)
img_tensor = transform(img_pil).unsqueeze(0)

# 3. 모델 추론
model.eval()
with torch.no_grad():
    output = model(img_tensor)
    probabilities = F.softmax(output[0], dim=0)
    prediction = torch.argmax(output, dim=1).item()
    confidence = probabilities[prediction].item() * 100

# --- 추가된 출력 부분 ---
labels = ["알", "백묵병", "응애", "정상"]
print("-" * 30)
print(f"분석 결과: {labels[prediction]}")
print(f"확신도(Confidence): {confidence:.2f}%")
print("-" * 30)
# ----------------------

# 4. 한글 및 퍼센트 출력 준비 (이미지용)
fontpath = "malgun.ttf" 
font = ImageFont.truetype(fontpath, 40)
draw = ImageDraw.Draw(img_pil)
text = f"{labels[prediction]} ({confidence:.1f}%)"
draw.text((50, 50), text, font=font, fill=(0, 255, 0))

# 5. OpenCV로 결과 출력
img_result = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
cv2.imshow('Bee Detection', img_result)
cv2.waitKey(0)
cv2.destroyAllWindows()

------------------------------
분석 결과: 알
확신도(Confidence): 100.00%
------------------------------


# -- 백묵병 분류

In [23]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import ImageFont, ImageDraw, Image

# 1. 이미지 로드 (이 셀 내에서 한꺼번에 처리)
image_path = 'test2.jpg' 
img_cv = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
img_pil = Image.fromarray(img_rgb)

# 2. 전처리 (이전 셀에서 정의한 transform 사용)
img_tensor = transform(img_pil).unsqueeze(0)

# 3. 모델 추론
model.eval()
with torch.no_grad():
    output = model(img_tensor)
    probabilities = F.softmax(output[0], dim=0)
    prediction = torch.argmax(output, dim=1).item()
    confidence = probabilities[prediction].item() * 100

# --- 추가된 출력 부분 ---
labels = ["알", "백묵병", "응애", "정상"]
print("-" * 30)
print(f"분석 결과: {labels[prediction]}")
print(f"확신도(Confidence): {confidence:.2f}%")
print("-" * 30)
# ----------------------

# 4. 한글 및 퍼센트 출력 준비 (이미지용)
fontpath = "malgun.ttf" 
font = ImageFont.truetype(fontpath, 40)
draw = ImageDraw.Draw(img_pil)
text = f"{labels[prediction]} ({confidence:.1f}%)"
draw.text((50, 50), text, font=font, fill=(0, 255, 0))

# 5. OpenCV로 결과 출력
img_result = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
cv2.imshow('Bee Detection', img_result)
cv2.waitKey(0)
cv2.destroyAllWindows()

------------------------------
분석 결과: 백묵병
확신도(Confidence): 100.00%
------------------------------


# -- 정상 분류

In [24]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import ImageFont, ImageDraw, Image

# 1. 이미지 로드 (이 셀 내에서 한꺼번에 처리)
image_path = 'test1.jpg' 
img_cv = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
img_pil = Image.fromarray(img_rgb)

# 2. 전처리 (이전 셀에서 정의한 transform 사용)
img_tensor = transform(img_pil).unsqueeze(0)

# 3. 모델 추론
model.eval()
with torch.no_grad():
    output = model(img_tensor)
    probabilities = F.softmax(output[0], dim=0)
    prediction = torch.argmax(output, dim=1).item()
    confidence = probabilities[prediction].item() * 100

# --- 추가된 출력 부분 ---
labels = ["알", "백묵병", "응애", "정상"]
print("-" * 30)
print(f"분석 결과: {labels[prediction]}")
print(f"확신도(Confidence): {confidence:.2f}%")
print("-" * 30)
# ----------------------

# 4. 한글 및 퍼센트 출력 준비 (이미지용)
fontpath = "malgun.ttf" 
font = ImageFont.truetype(fontpath, 40)
draw = ImageDraw.Draw(img_pil)
text = f"{labels[prediction]} ({confidence:.1f}%)"
draw.text((50, 50), text, font=font, fill=(0, 255, 0))

# 5. OpenCV로 결과 출력
img_result = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
cv2.imshow('Bee Detection', img_result)
cv2.waitKey(0)
cv2.destroyAllWindows()

------------------------------
분석 결과: 정상
확신도(Confidence): 99.96%
------------------------------


# -- 알 분류

In [25]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import ImageFont, ImageDraw, Image

# 1. 이미지 로드 (이 셀 내에서 한꺼번에 처리)
image_path = 'test3.jpg' 
img_cv = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
img_pil = Image.fromarray(img_rgb)

# 2. 전처리 (이전 셀에서 정의한 transform 사용)
img_tensor = transform(img_pil).unsqueeze(0)

# 3. 모델 추론
model.eval()
with torch.no_grad():
    output = model(img_tensor)
    probabilities = F.softmax(output[0], dim=0)
    prediction = torch.argmax(output, dim=1).item()
    confidence = probabilities[prediction].item() * 100

# --- 추가된 출력 부분 ---
labels = ["알", "백묵병", "응애", "정상"]
print("-" * 30)
print(f"분석 결과: {labels[prediction]}")
print(f"확신도(Confidence): {confidence:.2f}%")
print("-" * 30)
# ----------------------

# 4. 한글 및 퍼센트 출력 준비 (이미지용)
fontpath = "malgun.ttf" 
font = ImageFont.truetype(fontpath, 40)
draw = ImageDraw.Draw(img_pil)
text = f"{labels[prediction]} ({confidence:.1f}%)"
draw.text((50, 50), text, font=font, fill=(0, 255, 0))

# 5. OpenCV로 결과 출력
img_result = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
cv2.imshow('Bee Detection', img_result)
cv2.waitKey(0)
cv2.destroyAllWindows()

------------------------------
분석 결과: 알
확신도(Confidence): 100.00%
------------------------------


In [14]:
import cv2
import torch
import torch.nn.functional as F
from PIL import ImageFont, ImageDraw, Image
import numpy as np
from collections import deque

# 1. 설정 및 초기화
# 웹캠이면 0, 영상 파일이면 '파일경로.mp4' 입력
video_source = 0  # <--- 여기에 파일명을 넣으세요. 예: 'bee_video.mp4'
cap = cv2.VideoCapture('정상vedio.mp4') 

# 한글 폰트 설정 (폰트 파일 경로 확인!)
fontpath = "malgun.ttf" 
font = ImageFont.truetype(fontpath, 30)
labels = ["알", "백묵병", "응애", "정상"]

# 평균을 내기 위한 버퍼 (최근 10개 프레임 저장)
prob_buffer = deque(maxlen=10)

model.eval()

print("분석을 시작합니다. 'q'를 누르면 종료됩니다.")

# 1. 이전 결과들을 저장할 바구니 (최근 15프레임)
history = deque(maxlen=15) 

while True:
    ret, frame = cap.read()
    if not ret: break

    # 2. 전처리
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    img_tensor = transform(img_pil).unsqueeze(0)

    # 3. 모델 추론
    with torch.no_grad():
        output = model(img_tensor)
        probs = F.softmax(output[0], dim=0).cpu().numpy()
        history.append(probs) # 바구니에 저장

        # 2. 핵심: 최근 15개 프레임의 평균 확률 계산
        avg_probs = np.mean(history, axis=0)
        prediction = np.argmax(avg_probs)
        confidence = avg_probs[prediction] * 100
    
        # 3. 화면에 출력 (안정된 라벨)
        img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        draw = ImageDraw.Draw(img_pil)
        
        # 이제 라벨이 튀지 않고 평균적으로 확신하는 것만 나옵니다.
        text = f"{labels[prediction]} ({confidence:.1f}%)"
        draw.text((50, 50), text, font=font, fill=(0, 255, 0))
    
        cv2.imshow('Stable Analysis', cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR))
        if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()
print("분석이 종료되었습니다.")

분석을 시작합니다. 'q'를 누르면 종료됩니다.
분석이 종료되었습니다.


# 경고창

In [26]:
import cv2
import torch
import torch.nn.functional as F
import numpy as np
from PIL import ImageFont, ImageDraw, Image
from collections import deque

# 1. 환경 설정
cap = cv2.VideoCapture('백묵병vedio.mp4')
fgbg = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=30)
labels = ["알", "백묵병", "응애", "정상"]
fontpath = "malgun.ttf"
font = ImageFont.truetype(fontpath, 30)

# 경고 및 카운팅 변수
alert_threshold = 90.0
consecutive_frames = 10
warning_counter = 0
bee_count = 0
prev_centers = []

model.eval()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # --- [기능 2] 모델 분류 및 경고 알림 ---
    img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    img_tensor = transform(img_pil).unsqueeze(0)
    
    with torch.no_grad():
        output = model(img_tensor)
        probs = F.softmax(output[0], dim=0).cpu().numpy()
        pred_idx = np.argmax(probs)
        confidence = probs[pred_idx] * 100

    # 경고 로직 (응애/백묵병 90% 이상 연속 10프레임)
    if (labels[pred_idx] in ["응애", "백묵병"]) and (confidence >= alert_threshold):
        warning_counter += 1
    else:
        warning_counter = 0

    # --- [기능 3] 화면 시각화 ---
    draw = ImageDraw.Draw(img_pil)
    # 상태 텍스트
    draw.text((50, 50), f"상태: {labels[pred_idx]} ({confidence:.1f}%)", font=font, fill=(0, 255, 0))

    # 경고 테두리
    if warning_counter >= consecutive_frames:
        draw.rectangle([0, 0, frame.shape[1]-1, frame.shape[0]-1], outline="red", width=10)
        draw.text((50, 150), "!!! 위험 감지 !!!", font=font, fill="red")

    # 출력
    cv2.imshow('Integrated System', cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR))
    if cv2.waitKey(30) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [35]:
import cv2
import torch
import torch.nn.functional as F
import numpy as np
from PIL import ImageFont, ImageDraw, Image
from collections import deque

# 1. 환경 설정
video_path = '정상vedio.mp4'
cap = cv2.VideoCapture(video_path)

# --- [영상 저장을 위한 설정 추가] ---
output_path = '정상vedio(결과).mp4'
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*'mp4v') # mp4 파일 저장 코덱
writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
# ------------------------------------

labels = ["알", "백묵병", "응애", "정상"]
fontpath = "malgun.ttf"
font = ImageFont.truetype(fontpath, 30)

# 경고 및 카운팅 변수
alert_threshold = 90.0
consecutive_frames = 10
warning_counter = 0

model.eval()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # --- [기능 2] 모델 분류 ---
    img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    img_tensor = transform(img_pil).unsqueeze(0)
    
    with torch.no_grad():
        output = model(img_tensor)
        probs = F.softmax(output[0], dim=0).cpu().numpy()
        pred_idx = np.argmax(probs)
        confidence = probs[pred_idx] * 100

    # 경고 로직 (응애/백묵병 90% 이상)
    if (labels[pred_idx] in ["응애", "백묵병"]) and (confidence >= alert_threshold):
        warning_counter += 1
    else:
        warning_counter = 0

    # --- [기능 3] 화면 시각화 ---
    draw = ImageDraw.Draw(img_pil)
    draw.text((50, 50), f"상태: {labels[pred_idx]} ({confidence:.1f}%)", font=font, fill=(0, 255, 0))

    if warning_counter >= consecutive_frames:
        draw.rectangle([0, 0, width-1, height-1], outline="red", width=10)
        draw.text((50, 150), "!!! 위험 감지 !!!", font=font, fill="red")

    # PIL -> OpenCV(BGR) 변환
    display_frame = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
    
    # [영상 저장]
    writer.write(display_frame)
    
    # 화면 표시
    cv2.imshow('Integrated System', display_frame)
    
    if cv2.waitKey(30) & 0xFF == ord('q'): 
        break

# 자원 해제
cap.release()
writer.release() # 반드시 추가해야 영상이 깨지지 않고 저장됩니다.
cv2.destroyAllWindows()

print(f"분석이 완료되었습니다. 저장된 파일: {output_path}")

분석이 완료되었습니다. 저장된 파일: 정상vedio(결과).mp4
